# IDD Segmentation 20k → train at full resolution

**Vision-Based Lane Inference for Indian Roads · Phase 4**

Trains the segmentation network on the full IDD Segmentation dataset instead of
IDD-Lite. Nothing touches your laptop; the 24 GB lands on Colab's own disk and a
compact working copy is archived to Drive.

**Why.** The current model is trained on 1,403 images at 320×227. IDD
Segmentation 20k has 14k training images at full resolution. The three weakest
measured numbers are all starved by that:

| | current | cause |
|---|---|---|
| non-drivable IoU | 0.42 | 2.2% of pixels, 1,403 images |
| roadside-objects IoU | 0.48 | same |
| structured-road detection | 22.5% | a 150 mm marking is sub-pixel at 227 px tall |

**Order of work.** Sections 1–4 are done once. After that, start at section 5 —
the dataset is restored from Drive and never re-downloaded.

`Runtime → Change runtime type → T4 GPU` first.

## 1. Runtime, disk, Drive

In [ ]:
import shutil, subprocess, pathlib, torch

print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'NO GPU')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime > Change runtime type > T4 GPU.')

free = shutil.disk_usage('/content').free / 1e9
print('Colab local disk free : %.0f GB' % free)
print('  25.8 GB archives + ~40 GB extracted; the archives are deleted as soon')
print('  as extraction succeeds, which is what keeps this inside ~112 GB.')
if free < 65:
    raise RuntimeError('Not enough local disk. Runtime > Disconnect and delete '
                       'runtime, then reconnect.')

from google.colab import drive
drive.mount('/content/drive')
DEST = pathlib.Path('/content/drive/MyDrive/ODP'); DEST.mkdir(parents=True, exist_ok=True)
dfree = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
print('Drive free            : %.1f GB   (need ~6)' % dfree)
if dfree < 7:
    raise RuntimeError('Under 7 GB free on Drive. Clear space, or set '
                       'SKIP_ARCHIVE = True in section 4 and retrain in one session.')
print('\nready')

## 2. Download

IDD's download page redirects to a **pre-signed AWS S3 link**, and that link
carries its own signature — no cookie or login header is needed once you have
it. So paste the S3 URLs, not the `idd.insaan.iiit.ac.in` ones.

### Getting each URL

1. Logged in at https://idd.insaan.iiit.ac.in/dataset/download/
2. Click **IDD Segmentation (IDD 20k Part I)** — the browser starts downloading
3. **Cancel it immediately**
4. `chrome://downloads` → right-click the cancelled item → **Copy link address**.
   It should begin `https://indian-roads.s3.amazonaws.com/...` and contain
   `X-Amz-Signature`
5. Repeat for **Part II**

Signed links last 24 hours from generation. If one fails, regenerate on that page
and re-copy. Part II is optional — Part I alone is 10k images and already a large
step up from IDD-Lite.

In [ ]:
# Pre-signed S3 links, verified 24 Aug: Part I is 19.9 GB and Part II 5.9 GB,
# both gzip, both signed until roughly 23:45 IST today. If the pre-flight below
# reports an expired link, regenerate it at
# https://idd.insaan.iiit.ac.in/dataset/download/ , start the download in the
# browser, cancel it, and copy the s3.amazonaws.com address from the download
# manager - not the idd.insaan.iiit.ac.in page link, which needs a login cookie.
PART1 = 'PASTE_YOUR_OWN_IDD_DOWNLOAD_URL_HERE  # register at idd.insaan.iiit.ac.in and copy the link it gives you; the links are personal and expire in 24 h'
PART2 = 'PASTE_YOUR_OWN_IDD_DOWNLOAD_URL_HERE  # register at idd.insaan.iiit.ac.in and copy the link it gives you; the links are personal and expire in 24 h'

import pathlib, subprocess, re

RAW = pathlib.Path('/content/idd_raw'); RAW.mkdir(parents=True, exist_ok=True)
TARGETS = [('part1', PART1, 19.9), ('part2', PART2, 5.9)]

def check(url, name):
    if not url.strip():
        return False
    if 'X-Amz-Signature' not in url:
        raise RuntimeError(
            '%s does not look like a pre-signed S3 link. It should start with\n'
            '  https://indian-roads.s3.amazonaws.com/...\n'
            'and contain X-Amz-Signature. You may have copied the '
            'idd.insaan.iiit.ac.in page link instead - that one needs a login '
            'cookie. Start the download in the browser, cancel it, and copy the '
            'link from chrome://downloads.' % name)
    return True

def fetch(name, url, gb):
    out = RAW / (name + '.bin')
    if out.exists() and out.stat().st_size > 0.9 * gb * 1e9:
        print('%s: already have %.1f GB' % (name, out.stat().st_size/1e9)); return
    # Pre-flight 512 bytes, piped through head so an unsigned or expired link
    # cannot stream gigabytes into memory before failing.
    probe = subprocess.run(
        'curl -sL --max-time 45 -r 0-511 %s 2>/dev/null | head -c 512' % repr(url),
        shell=True, capture_output=True).stdout
    if not (probe[:2] in (b'PK', b'\x1f\x8b') or probe[257:262] == b'ustar'):
        raise RuntimeError('%s: server did not return an archive. It sent:\n  %s\n'
                           'The link has probably expired - regenerate it.'
                           % (name, probe[:200].decode('utf-8','replace')))
    print('%s: pre-flight OK, fetching ~%s GB ...' % (name, gb), flush=True)
    r = subprocess.run(['curl','-L','-C','-','--retry','8','--retry-delay','15',
                        '--retry-all-errors','--progress-bar','-o',str(out),url])
    if r.returncode != 0:
        raise RuntimeError('curl exited %d. Re-run this cell; it resumes.' % r.returncode)
    print('%s: %.2f GB' % (name, out.stat().st_size/1e9))

any_ok = False
for name, url, gb in TARGETS:
    if check(url, name):
        fetch(name, url, gb); any_ok = True
if not any_ok:
    raise RuntimeError('No URLs given. Paste at least PART1.')
!ls -lh /content/idd_raw && df -h /content | tail -1

## 3. Extract and rasterise the labels

In [ ]:
import pathlib, zipfile, tarfile

RAW = pathlib.Path('/content/idd_raw')
SEG = pathlib.Path('/content/idd_full'); SEG.mkdir(parents=True, exist_ok=True)

def unpack(path, dest):
    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z: z.extractall(dest)
    elif tarfile.is_tarfile(path):
        with tarfile.open(path) as t: t.extractall(dest)
    else:
        raise RuntimeError(path.name + ': not zip or tar: ' + repr(path.read_bytes()[:80]))

for f in sorted(RAW.glob('*.bin')):
    print('extracting', f.name, '...', flush=True)
    unpack(f, SEG)

def find_root(base):
    for p in [base] + [d for d in base.rglob('*') if d.is_dir()]:
        if (p/'leftImg8bit').is_dir() and (p/'gtFine').is_dir():
            return p
    return None

SRC = find_root(SEG)
if SRC is None:
    print('Tree under', SEG, '(first 40):')
    for n,q in enumerate(sorted(SEG.rglob('*'))):
        if n>=40: break
        print('  ', q.relative_to(SEG))
    raise RuntimeError('leftImg8bit/ + gtFine/ not found - see tree above')

print('\nroot:', SRC)
for split in ('train','val','test'):
    d = SRC/'leftImg8bit'/split
    print('  %-6s %d images' % (split, len(list(d.glob('*/*'))) if d.is_dir() else 0))
print('  gtFine polygons: %d' % len(list((SRC/'gtFine').rglob('*_polygons.json'))))

# The archives are 25.8 GB and extraction has succeeded, so release them now.
# Colab's disk is ~112 GB and the extracted tree plus archives together come
# close enough to fill it that later stages fail on space.
import shutil
shutil.rmtree(RAW, ignore_errors=True)
print()
!df -h /content | tail -1

IDD ships ground truth as polygon JSON, not label images — they must be
rasterised with IDD's own tooling. `createLabels.py` can emit the 7-class
level-1 hierarchy directly (`--id-type level1Id`), which is the hierarchy this
project uses, so no class remapping is needed. If that id type is unavailable in
your copy of the tooling the cell falls back to level-3 and maps down using the
official table from `anue_labels.py`.

In [ ]:
import pathlib, subprocess, sys, os

# createLabels.py needs numpngw, which Colab does not preinstall.
subprocess.run([sys.executable,'-m','pip','install','-q','numpngw'], check=False)

CODE = pathlib.Path('/content/public-code')
if not CODE.is_dir():
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/AutoNUE/public-code.git', str(CODE)], check=True)

# IDD's tooling dates from 2018 and guards its imports with
#     try: from PIL import PILLOW_VERSION
#     except: print("Please install the module 'Pillow'"); sys.exit(-1)
# PILLOW_VERSION was removed in Pillow 9.0, so on any current Pillow the check
# fails and the script exits reporting a missing package that is in fact
# installed and working. Rewrite it to the attribute that still exists.
for f in CODE.rglob('*.py'):
    t = f.read_text(errors='ignore')
    if 'PILLOW_VERSION' in t:
        f.write_text(t.replace('PILLOW_VERSION', '__version__'))
        print('patched Pillow check in', f.relative_to(CODE))

# json2labelImg.py imports anue_labels as a flat module and it lives in
# helpers/, so helpers/ and preperation/ must both be importable.
env = dict(os.environ, ANUE=str(SRC),
           PYTHONPATH=os.pathsep.join([str(CODE), str(CODE/'helpers'),
                                       str(CODE/'preperation')]))

def generate(id_type):
    r = subprocess.run([sys.executable, 'createLabels.py',
                        '--datadir', str(SRC), '--id-type', id_type,
                        '--num-workers', '8'],
                       cwd=str(CODE/'preperation'),
                       env=env, capture_output=True, text=True)
    made = len(list((SRC/'gtFine').rglob('*label%ss.png' % id_type)))
    print('%s -> rc %d, %d masks' % (id_type, r.returncode, made))
    if made == 0:
        print(((r.stdout or '') + (r.stderr or ''))[-1200:])
    return made

LEVEL = 'level1Id'
if generate(LEVEL) == 0:
    print('\nlevel1Id produced nothing, falling back to level3Id')
    LEVEL = 'level3Id'
    if generate(LEVEL) == 0:
        raise RuntimeError('Label generation failed - see output above.')
print('\nusing', LEVEL)

## 4. Downscale once and archive

Full-resolution IDD is 1920×1080. Training at that size on a T4 is neither
necessary nor affordable; 512×288 keeps 2.3× the linear resolution of IDD-Lite
while training in a fraction of the time.

Labels are resized nearest-neighbour and images by area — interpolating a label
map invents classes that were never annotated.

In [ ]:
import cv2, numpy as np, pathlib
from concurrent.futures import ThreadPoolExecutor

OUT  = pathlib.Path('/content/idd_work'); OUT.mkdir(parents=True, exist_ok=True)
SIZE = (512, 288)         # (W, H)
SKIP_ARCHIVE = False      # set True if Drive is tight; then train in this session

# Official level3Id -> level1Id table, from AutoNUE/public-code anue_labels.py.
# Only used if level1Id masks were unavailable.
L3_TO_L1 = {0:0, 1:0, 2:1, 3:1, 4:2, 5:2,
            6:3, 7:3, 8:3, 9:3, 10:3, 11:3, 12:3,
            13:4, 14:4, 15:4, 16:4, 17:4, 18:4, 19:4, 20:4, 21:4,
            22:5, 23:5, 24:5, 25:6}
LUT = np.full(256, 255, np.uint8)
if LEVEL == 'level3Id':
    for k, v in L3_TO_L1.items(): LUT[k] = v
else:
    for k in range(7): LUT[k] = k

SUFFIX = '_gtFine_label%ss.png' % LEVEL

def convert(args):
    img_path, split = args
    stem = img_path.name.split('_leftImg8bit')[0]
    gtdir = pathlib.Path(str(img_path.parent).replace('leftImg8bit','gtFine'))
    lbl = gtdir / (stem + SUFFIX)
    if not lbl.is_file():
        hits = list(gtdir.glob(stem + '*label*.png'))
        if not hits: return 0
        lbl = hits[0]
    im = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    lb = cv2.imread(str(lbl), cv2.IMREAD_GRAYSCALE)
    if im is None or lb is None: return 0
    im = cv2.resize(im, SIZE, interpolation=cv2.INTER_AREA)
    lb = LUT[cv2.resize(lb, SIZE, interpolation=cv2.INTER_NEAREST)]
    drive = img_path.parent.name
    di = OUT/'leftImg8bit'/split/drive; di.mkdir(parents=True, exist_ok=True)
    dl = OUT/'gtFine'/split/drive;      dl.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(di/(stem+'_image.jpg')), im, [cv2.IMWRITE_JPEG_QUALITY, 92])
    cv2.imwrite(str(dl/(stem+'_label.png')), lb)
    return 1

total = 0
for split in ('train','val'):
    d = SRC/'leftImg8bit'/split
    if not d.is_dir(): continue
    files = [(p, split) for p in sorted(d.glob('*/*')) if p.suffix.lower() in ('.png','.jpg')]
    with ThreadPoolExecutor(8) as ex:
        n = sum(ex.map(convert, files))
    print('%-6s %d/%d converted' % (split, n, len(files)))
    total += n
if total == 0:
    raise RuntimeError('Nothing converted - the label suffix did not match.')
print('total', total)
!du -sh /content/idd_work

In [ ]:
# Labels must be 0-6 (+255), and class 0 must actually be road.
import glob, numpy as np, cv2, matplotlib.pyplot as plt
lbls = sorted(glob.glob('/content/idd_work/gtFine/train/*/*_label.png'))[:300]
u = set()
for p in lbls: u |= set(np.unique(cv2.imread(p, 0)).tolist())
print('label values:', sorted(u))
assert u <= set(range(7)) | {255}, 'unexpected label ids: %s' % sorted(u)

PAL = np.array([[220,60,60],[255,165,0],[255,0,255],[0,0,255],
                [0,255,255],[60,200,60],[160,200,240]], np.uint8)
imgs = sorted(glob.glob('/content/idd_work/leftImg8bit/train/*/*_image.jpg'))
fig, ax = plt.subplots(2, 3, figsize=(16, 6))
for i, p in enumerate(imgs[::max(1,len(imgs)//3)][:3]):
    im = cv2.imread(p)[:, :, ::-1]
    lb = cv2.imread(p.replace('leftImg8bit','gtFine').replace('_image.jpg','_label.png'), 0)
    col = np.zeros((*lb.shape, 3), np.uint8)
    for k in range(7): col[lb == k] = PAL[k]
    ax[0][i].imshow(im); ax[0][i].axis('off'); ax[0][i].set_title('image')
    ax[1][i].imshow((0.5*im + 0.5*col).astype('uint8')); ax[1][i].axis('off')
    ax[1][i].set_title('labels (red = drivable)')
plt.tight_layout(); plt.show()
print('If red is not the road surface, the class mapping is wrong - stop here.')

In [ ]:
# Archive the working copy, then reclaim the 24 GB of originals.
import subprocess, shutil, pathlib
if not SKIP_ARCHIVE:
    dest = pathlib.Path('/content/drive/MyDrive/ODP')
    subprocess.run(['tar','-czf','/content/idd_work.tar.gz','-C','/content','idd_work'], check=True)
    pack = pathlib.Path('/content/idd_work.tar.gz')
    shutil.copy(pack, dest/'idd_work.tar.gz')
    print('archived %.2f GB to %s' % (pack.stat().st_size/1e9, dest/'idd_work.tar.gz'))
    pack.unlink()

shutil.rmtree('/content/idd_raw', ignore_errors=True)
shutil.rmtree('/content/idd_full', ignore_errors=True)
!df -h /content | tail -1

## 5. Train

**Start here on later runs.** The dataset is restored from Drive, so sections
1–4 are never repeated.

In [ ]:
import pathlib, subprocess, zipfile
from google.colab import drive
try: drive.mount('/content/drive')
except Exception: pass
dest = pathlib.Path('/content/drive/MyDrive/ODP')

if not pathlib.Path('/content/idd_work').is_dir():
    print('restoring dataset from Drive ...', flush=True)
    subprocess.run(['tar','-xzf',str(dest/'idd_work.tar.gz'),'-C','/content'], check=True)

if not pathlib.Path('/content/odp/lane_inference').is_dir():
    pathlib.Path('/content/odp').mkdir(exist_ok=True)
    with zipfile.ZipFile(dest/'odp_bundle.zip') as z: z.extractall('/content/odp')

import glob
print('code    :', pathlib.Path('/content/odp/lane_inference/main.py').is_file())
print('train   :', len(glob.glob('/content/idd_work/leftImg8bit/train/*/*_image.jpg')))
print('val     :', len(glob.glob('/content/idd_work/leftImg8bit/val/*/*_image.jpg')))

In [ ]:
# Lane-line pseudo-labels from the YOLOP teacher, for the distilled head.
!cd /content/odp/lane_inference && pip install -q prefetch_generator yacs 2>&1 | tail -1
!cd /content/odp/lane_inference && PYTHONPATH=. python -u -m scripts.make_lane_pseudolabels \
    --root /content/idd_work --splits train,val 2>&1 | tail -4

In [ ]:
# Checkpoints are written STRAIGHT TO DRIVE, not to Colab's local disk.
#
# This matters more than it looks. Colab's /content is destroyed when the
# session ends, and free-tier GPU sessions end without warning when the quota
# runs out. A run that stored last.pt locally and copied it to Drive only after
# training finished lost 26 epochs of a 40-epoch schedule exactly that way -
# and --resume could not help, because the file it resumes from died with the
# session. Writing to Drive as training goes makes --resume actually work.
#
# ~2.5 min/epoch on a T4 with 7k images at 288x512. Re-run this cell after any
# disconnect; it picks up from the last completed epoch.
!cd /content/odp/lane_inference && PYTHONPATH=. python -u -m models.train \
    --epochs 40 \
    --batch-size 16 \
    --lr 5e-4 \
    --input-size 288x512 \
    --num-workers 2 \
    --lane-head \
    --resume \
    --run-name seg_idd20k \
    --checkpoint-dir /content/drive/MyDrive/ODP/checkpoints \
    --data-root /content/idd_work

In [ ]:
# Checkpoints are already on Drive, so this only summarises and downloads.
import json, pathlib, shutil, zipfile
out  = pathlib.Path('/content/drive/MyDrive/ODP/checkpoints/seg_idd20k')
if not out.is_dir():
    raise SystemExit('No checkpoints at %s - has training run?' % out)
print('files on Drive:', sorted(p.name for p in out.iterdir()))

h = json.loads((out/'history.json').read_text())
b = max(h, key=lambda x: x['val_miou'])
print('\nepochs completed : %d' % len(h))
print('best epoch       : %d' % b['epoch'])
print('mIoU             : %.4f' % b['val_miou'])
print('drivable IoU     : %.4f' % b['val_drivable_iou'])
print('lane-line IoU    : %.4f' % b.get('val_lane_iou', float('nan')))
print('pixel accuracy   : %.4f' % b['val_pixel_acc'])
print('\nIDD-Lite baseline: mIoU 0.6833, drivable 0.9326, lane 0.5480')

pack = pathlib.Path('/content/seg_idd20k.zip')
with zipfile.ZipFile(pack, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in out.iterdir():
        if f.is_file():
            z.write(f, pathlib.Path('checkpoints/seg_idd20k')/f.name)
print('\npackaged %.1f MB' % (pack.stat().st_size/1e6))
from google.colab import files; files.download(str(pack))